In [16]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
from sklearn.datasets import make_regression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.utils.data.dataset import TensorDataset

In [29]:
class MyDataModule(pl.LightningDataModule):
    def __init__(self):
        super().__init__()
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def prepare_data(self):
        X, y, coef = make_regression(n_samples=5000, n_features=20, random_state=42, coef=True)
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2)

        X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)

        self.train_dataset = TensorDataset(X_train, y_train)
        self.val_dataset = TensorDataset(X_val, y_val)
        self.test_dataset = TensorDataset(X_test, y_test)

    def setup(self, stage: str):
        pass

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=256)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=32)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=256)

    # def predict_dataloader(self):
    #     return DataLoader(self.mnist_predict, batch_size=32)

In [18]:
class LitModel(pl.LightningModule):
    def __init__(self, input_size, hidden_size, output_dim, lr=0.001):
        super().__init__()
        self.save_hyperparameters()  # Call this first to save __init__ arguments to self.hparams

        self.layer1 = nn.Linear(self.hparams.input_size, self.hparams.hidden_size)
        self.layer2 = nn.Linear(self.hparams.hidden_size, self.hparams.hidden_size)
        self.layer3 = nn.Linear(self.hparams.hidden_size, self.hparams.output_dim)
        self.criterion = nn.MSELoss()

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.layer3(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)
        r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_r2", r2, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)
        r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
        self.log("test_loss", loss)
        self.log("test_r2", r2)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


In [36]:
def quick_train_model(config):
    model = LitModel(
        input_size=20,
        hidden_size=config['hidden_size'],
        output_dim=1,
        lr=config['lr']
    )
    trainer = Trainer(
        max_epochs=20,
        accelerator='gpu',
        devices=1,
        callbacks=[TuneReportCallback({'loss': 'val_loss'}, on='validation_end')],
    )
    trainer.fit(model, datamodule)


In [40]:
analysis = tune.run(
    quick_train_model,
    resources_per_trial={"gpu": 1},
    config={
        'hidden_size': tune.choice([64, 128, 256]),
        'lr': tune.loguniform(1e-4, 1e-2)
    },
    num_samples=10
)


2026-06-17 20:55:18,555	INFO tune.py:615 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(pid=73145) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/ray/_private/function_manager.py:677: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
(pid=73145)   actor_class = pickle.loads(pickled_class)
(quick_train_model pid=73145) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(quick_train_model pid=73145) GPU available: True (cuda), used: True
(quick_train_model pid=73145) TPU available: False, using: 0 TPU cores
(quick_train_model pid=73145) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments

Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 13/13 [00:00<00:00, 78.78it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:  76%|███████▌  | 19/25 [00:00<00:00, 557.72it/s]
(quick_train_model pid=73145) 
Validation DataLoader 0:  80%|████████  | 20/25 [00:00<00:00, 558.89it/s]


(quick_train_model pid=73145) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
(quick_train_model pid=73145) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
(quick_train_model pid=73145) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Trial name,loss
quick_train_model_15973_00000,23440.9
quick_train_model_15973_00001,104.436
quick_train_model_15973_00002,22779.1
quick_train_model_15973_00003,81.316
quick_train_model_15973_00004,22665.1
quick_train_model_15973_00005,7836.1
quick_train_model_15973_00006,24870
quick_train_model_15973_00007,23457.8
quick_train_model_15973_00008,25122.3
quick_train_model_15973_00009,16.9724


...

(quick_train_model pid=73792) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
(quick_train_model pid=73792) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
(quick_train_model pid=73792) /home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


(quick_train_model pid=73792) 
Epoch 1: 100%|██████████| 13/13 [00:00<00:00, 271.05it/s, v_num=0, val_loss=2.02e+4, val_r2=0.124]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:  32%|███▏      | 8/25 [00:00<00:00, 546.49it/s]
(quick_train_model pid=73792) 
Validation DataLoader 0:  36%|███▌      | 9/25 [00:00<00:00, 547.80it/s]
(quick_train_model pid=73792) 
Validation DataLoader 0:  40%|████      | 10/25 [00:00<00:00, 543.67it/s]
(quick_train_model pid=73792) 
Validation DataLoader 0:  44%|████▍     | 11/25 [00:00<00:00, 543.64it/s]
(quick_train_model pid=73792) 
Validation DataLoader 0:  48%|████▊     | 12/25 [00:00<00:00, 540.86it/s]
(quick_train_model pid=73792) 
(quick_train_model pid=73792) 
Epoch 2: 100%|██████████| 13/13 [00:00<00:00, 282.45it/s, v_num=0, val_loss=4.53e+3, val_r2=0.804]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:  48%|███

2026-06-17 20:56:51,719	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/home/dom/ray_results/quick_train_model_2026-06-17_20-55-18' in 0.0037s.
2026-06-17 20:56:51,725	INFO tune.py:1033 -- Total run time: 93.17 seconds (93.13 seconds for the tuning loop).


{'hidden_size': 128, 'lr': 0.006821727740033031}


(quick_train_model pid=73792) `Trainer.fit` stopped: `max_epochs=20` reached.


In [42]:
print(analysis.get_best_config(metric='loss', mode='min'))

{'hidden_size': 128, 'lr': 0.006821727740033031}


In [43]:
model = LitModel(input_size=20, hidden_size=128, output_dim=1, lr=0.007)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min',
    verbose=True
)

checkpoint_callback = ModelCheckpoint(
    dirpath='./checkpoints/',
    filename='model-{epoch:02d}-{val_loss:.2f}',
    monitor='val_loss',
    mode='min',
    save_top_k=3,
    save_last=True
)

trainer = Trainer(
    max_epochs=50,
    accelerator='gpu',
    devices=1,
    callbacks=[early_stop_callback, checkpoint_callback],
)
datamodule = MyDataModule()
trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/dom/GitRepos/algorithms/DL/Lab11/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type    | Params | Mode  | FLOPs
------------------------------------------------------
0 | layer1    | Linear  | 2.7 K  | train | 0    
1 | layer2    | Linear  | 16.5 K | train | 0    
2 | layer3    | Linear  | 129    | train | 0    
3 | criterion | MSELoss | 0      | train | 0    
------------------------------------------------------
19.3 K    Trainable params
0         Non-trainable params
19.3 K    Total

/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Conside

Epoch 0: 100%|██████████| 13/13 [00:00<00:00, 151.82it/s, v_num=9]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 13/13 [00:00<00:00, 60.90it/s, v_num=9, val_loss=1.94e+4, val_r2=0.179]

Metric val_loss improved. New best score: 19372.789


Epoch 1: 100%|██████████| 13/13 [00:00<00:00, 199.75it/s, v_num=9, val_loss=1.94e+4, val_r2=0.179]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 13/13 [00:00<00:00, 68.47it/s, v_num=9, val_loss=3.25e+3, val_r2=0.862] 

...

Metric val_loss improved by 0.076 >= min_delta = 0.0. New best score: 6.996
`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 13/13 [00:00<00:00, 67.87it/s, v_num=9, val_loss=7.000, val_r2=1.000]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 228.78it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            4.551116466522217
         test_r2            0.9998220801353455
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 4.551116466522217, 'test_r2': 0.9998220801353455}]